In [0]:
%pip install databricks-feature-engineering>=0.13.0 mlflow==3.5.0 protobuf==6.31.1

# Restart the Python interpreter for changes to take effect
dbutils.library.restartPython()

In [0]:
import os

def dev_lab_setup(catalog_name, schema_name=None):
    if schema_name is None:
        schema_name=os.path.basename(os.getcwd())
        schema_name=schema_name.replace('-','_')
    spark.sql(f"USE CATALOG {catalog_name}")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_name}")
    spark.sql(f"USE SCHEMA {schema_name}")

    print(f"Using catalog: {catalog_name}")
    print(f"Creating and using schema: {schema_name}") 
    return catalog_name, schema_name

In [0]:
from databricks.feature_engineering import FeatureLookup,FeatureEngineeringClient

import mlflow
from mlflow.types.utils import _infer_schema
from mlflow.models import infer_signature

from sklearn.tree import DecisionTreeClassifier

import time

import warnings

from databricks.sdk.service.serving import EndpointTag

from pyspark.sql.functions import col, when, trim


def initialize_uc(catalog_name, schema_name):

    print("Cleaning up old tables")
    table_name = 'telco_table'
    features_table_name = 'features'
    features_online_name = 'features_online'
    spark.sql(f"USE CATALOG {catalog_name}")
    spark.sql(f"USE SCHEMA {schema_name}")
    spark.sql(f"DROP TABLE IF EXISTS {table_name}")
    spark.sql(f"DROP TABLE IF EXISTS {features_table_name}")
    spark.sql(f"DROP TABLE IF EXISTS {features_table_name}")
    spark.sql(f"DROP TABLE IF EXISTS {features_online_name}")

    print(f'Using catalog {catalog_name} and schema {schema_name}.')

def data_preparation(
    databricks_share_name,
    catalog_name,
    schema_name
    ):

    # Load dataset with spark
    shared_volume_name = 'telco' # From Marketplace
    csv_name = 'telco-customer-churn-missing' # CSV file name
    dataset_p_telco = f"/Volumes/databricks_telco_customer_dataset/v01/telco/telco-customer-churn-missing.csv" # Full path
    # Dataset specs
    primary_key = "customerID"
    response = "Churn"
    features = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"] # Keeping numerical only for simplicity and demo purposes

    # Read dataset and handle empty/whitespace strings
    # Convert all fields to double for spark compatibility
    telco_df = spark.read.csv(dataset_p_telco, inferSchema=True, header=True, multiLine=True, escape='"')\
                .withColumn("TotalCharges", 
                           when((trim(col("TotalCharges")) == "") | (col("TotalCharges").isNull()), None)
                           .otherwise(col("TotalCharges").cast('double')))\
                .withColumn("SeniorCitizen", 
                           when((trim(col("SeniorCitizen")) == "") | (col("SeniorCitizen").isNull()), None)
                           .otherwise(col("SeniorCitizen").cast('double')))\
                .withColumn("tenure", 
                           when((trim(col("tenure")) == "") | (col("tenure").isNull()), None)
                           .otherwise(col("tenure").cast('double')))\
                .na.drop(how='any')

    # IMPORTANT: Cache the dataframe to materialize the transformations
    telco_df.count()  # Force execution to materialize the cache

    # Separate features and ground-truth
    features_df = telco_df.select(primary_key, *features)
    response_df = telco_df.select(primary_key, response)

    # Write features_df to an offline feature table
    feature_table_name = f"{catalog_name}.{schema_name}.features"
    response_table_name = f"{catalog_name}.{schema_name}.response"
    
    spark.sql(f'DROP TABLE IF EXISTS {feature_table_name}') # Drop existing feature table if it exists
    spark.sql(f'DROP TABLE IF EXISTS {response_table_name}') # Drop existing response table if it exists

    # Create feature table
    fe = FeatureEngineeringClient()
    fe.create_table(
        name=feature_table_name,
        df=features_df,
        primary_keys=[primary_key],
        description="Example feature table"
    )

    # Create response table
    response_df.write.mode("overwrite").saveAsTable(response_table_name)

    return f"Created features table in {feature_table_name}"